In [20]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.extract.data_extractor import DataExtractor
from src.transform.data_transformer import DataTransformer
from src.load.data_loader import DataLoader

extractor = DataExtractor()
transformer = DataTransformer()
loader = DataLoader()

raw_data = extractor.download_data()

loader.save_to_csv(raw_data,"../data/raw/market_data_raw.csv")

data = transformer.clean_data(raw_data)
transformer.validate_data(data)

data = transformer.calculate_daily_returns(data)
data = transformer.normalize_prices(data,extractor.tickers)
data = transformer.add_asset_class(data)
data = transformer.add_asset_name(data)

loader.save_to_csv(data,"../data/processed/market_data.csv")
loader.save_to_sqlite(data)



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Data saved to ../data/raw/market_data_raw.csv
Rows: 2751
Tickers: 10
Price
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
Ticker        str
dtype: object
Data saved to ../data/processed/market_data.csv
Data saved to ../data/database/market_data.db


In [21]:
volatility_df = (
    data.groupby(["Asset Name", "Asset Class"])["Daily Return"]
    .std()
    .reset_index()
)

volatility_df.rename(
    columns={"Daily Return": "Volatility"},
    inplace=True
)

volatility_df.to_csv(
    "../data/processed/volatility.csv",
    index=False
)